In [28]:
from pathlib import Path
import ims
from skimage.io import imshow
from sympy.strategies.branch import condition

import preprocess

# Step 0: Define all necessary directories.
input_dir = Path("./data/IMS_Olive_Oil_Geography/Test/")
output_dir = Path("./output/Jupyter_Test/")

# Check and create directories if necessary.
if not output_dir.exists():
    output_dir.mkdir(parents=True, exist_ok=True)

# Step 1: Preprocess the .mea files
# - RIP Interpolation (for each dataset)
# - Background noise reduction
# - Gaussian smoothing

# Load the test dataset and interpolate to a common RIP
dataset = ims.Dataset.read_mea(input_dir)
dataset.interp_riprel()

# Get the first test spectrum.
spectrum = dataset[0]

In [7]:
# =========================================== JUST PLOTTING THE CUTTED RAW SPECTRUM ===========================================
import plot
spectrum.cut_dt(start=1.02, stop=1.2)
spectrum.cut_rt(start=100, stop=140)

# Plot the raw spectrum.
plot.single_gcims_plot(spectrum=spectrum,
                       plot_dir=output_dir,
                       condition="raw_cut")


<Figure size 1000x600 with 0 Axes>

In [29]:
# =========================================== I. PREPROCESSING ===========================================
import plot
from scipy.ndimage import uniform_filter
# Plot the raw spectrum.
plot.single_gcims_plot(spectrum=spectrum,
                       plot_dir=output_dir,
                       condition="raw",
                       peak_cords=None)

spectrum.rip_ms = 1
# Baseline correction - removing the noisy background by subtracting the mean of an empty area.
spectrum.sub_first_rows(50)
spectrum.cut_dt(start=1.02, stop=1.2)
spectrum.cut_rt(start=100, stop=140)

#uniform_filter_size= (11, 11)
#spectrum.values = uniform_filter(input=spectrum.values, size=uniform_filter_size)
#from scipy.ndimage import gaussian_filter
#spectrum.values = gaussian_filter(spectrum.values, sigma=1.0)

# Smoothing the spectrum with the Savitzky–Golay filter.
spectrum.savgol(window_length=10, polyorder=4)

# Remove negative values after applying the smooth filter.
spectrum.values[spectrum.values < 0] = 0.0

# Plot the processed spectrum.

plot.single_gcims_plot(spectrum=spectrum,
                       plot_dir=output_dir,
                       condition="processed",
                       peak_cords=None)


<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

In [8]:
# =========================================== II. PEAK DETECTION - detect_peaks - Fast version ===========================================
import numpy as np
import pandas as pd
idx = 1
neighbor_distance = 5
create_heatmaps = False

print(f"- Peak Detection Started on: Spectrum {idx + 1}")
# Keep the non normalized values for later use.
values_non_normalized = spectrum.values.copy()
# Normalize the spectrum [0-1].
spectrum.normalization()
# List of dictionaries which contain selected peak information. This list is returned at the end of this function.
spectrum_peak_data = []

try:
    label = output_dir.name

    # 1. Check for peaks and put the peak coordinates in a list with tuples (Retention , Drift).
    # Define the min. peak threshold intensity [non normalized value].
    peak_threshold = 100
    # How many intensity values are there in total?
    intensity_count_max = spectrum.values.shape[0] * spectrum.values.shape[1]
    # How many intensities values are over the peak_threshold?
    threshold_intensity_count = len(np.where(values_non_normalized >= peak_threshold)[0])
    # Calculate the % of intensity values which are below the peak_threshold.
    threshold_percentile = 100 - (threshold_intensity_count / intensity_count_max) * 100

    # Peak intensity detection threshold.
    intensity_threshold = np.percentile(spectrum.values, [threshold_percentile])
    # Find peaks with the integrated find_peaks function.
    spectrum.detect_peaks(threshold_rel=intensity_threshold, peak_size=3)
    #spectrum.find_peaks(limit=peak_threshold, denoise=None)
    # List with peak coordinates as tuples [(Retention-index, Drift-index)].
    peak_coords = np.column_stack((spectrum.peaklist['y'].values, spectrum.peaklist['x'].values))
    #peak_coords = np.column_stack((spectrum.peak_table['y'].values, spectrum.peak_table['x'].values))

    if len(peak_coords) > 0:
        # 2. Remove peaks which are located near the spectrum boarders.
        # list with all peak retention indices.
        peak_ret_idx_list = peak_coords[:, 0]
        # list with all peak drift indices.
        peak_drift_idx_list = peak_coords[:, 1]
        # Integer with the maximum retention and drift index.
        ret_idx_max, drift_idx_max = spectrum.values.shape

        # Create single mask conditions.
        mask_ret_min = peak_ret_idx_list >= neighbor_distance
        mask_ret_max = peak_ret_idx_list < (ret_idx_max - neighbor_distance)
        mask_drift_min = peak_drift_idx_list >= neighbor_distance
        mask_drift_max = peak_drift_idx_list < (drift_idx_max - neighbor_distance)
        # Combine all 4 masks to one mask condition. This mask will remove all peaks which are too close to the boarder.
        # This will create a list with tuples of the same length as the found peak list (peak_coords).
        # If all single mask conditions are met, true will be saved in this list. Otherwise, it will be set to False.
        final_validity_mask = (mask_ret_min &
                               mask_ret_max &
                               mask_drift_min &
                               mask_drift_max)

        # Use the mask on the peak_coords list.
        # Only Coordinates where the mask is set to "True" are kept. The rest will be removed.
        peak_coords = peak_coords[final_validity_mask]

        # 3. Reload the non normalized values and crate a peak plot.
        # Reload the non normalized values.
        spectrum.values = values_non_normalized
        # Plot the current spectrum with the strongest peaks marked with a red X.
        plot.single_gcims_plot(spectrum=spectrum,
                               plot_dir=output_dir,
                               condition="peaks",
                               size=(12, 8),
                               peak_cords=peak_coords)

        # 4. Extend peaks. Each peak will be saved ....
        # Copy the spectrum.values and add a padding of 0 around them.
        # Why? If a peak is detected near the spectrum boarder, this could lead to errors in the shape and or saved
        # intensities of the peak windows. The padding guarantees the correct shape and failsafe the intensities of
        # indices outside the spectrum boarder to 0.
        padded_values = np.pad(
            spectrum.values,
            pad_width=neighbor_distance,
            mode='constant',
            constant_values=0
        )

        print(f" - Creating npy-files and heatmaps: Spectrum {idx + 1} [{spectrum.name}]")
        # Walk through all peak coordinates.
        for ret_idx, drift_idx in peak_coords:
            # Create the heatmap window.
            start_ret_idx = ret_idx - neighbor_distance
            end_ret_idx = ret_idx + 3 * neighbor_distance + 1
            start_drift_idx = drift_idx - neighbor_distance
            end_drift_idx = drift_idx + 3 * neighbor_distance + 1
            peak_window = padded_values[start_ret_idx:end_ret_idx, start_drift_idx:end_drift_idx]


            # Plot the peak heatmaps.
            if create_heatmaps:
                plot.plot_peak_heatmap(spectrum=spectrum,
                                       heatmap_window=peak_window,
                                       ret_idx=ret_idx,
                                       drift_idx=drift_idx,
                                       window_size=neighbor_distance,
                                       plot_dir=output_dir)


            # Check if the peak window has the correct shape. This should always be the case!
            if peak_window.shape != (4*neighbor_distance + 1, 4*neighbor_distance + 1):
                raise ValueError(f"Peak Window has the wrong shape: {peak_window.shape}. "
                                 f"Expected Shape: ({4*neighbor_distance + 1}, {4*neighbor_distance + 1})")

            # Define the numpy filename and create the needed directories.
            npy_dir = output_dir / "npy" / spectrum.name
            if not npy_dir.exists():
                npy_dir.mkdir(parents=True, exist_ok=True)

            npy_filename = npy_dir / f"PW_{spectrum.name}_RT{ret_idx}_DT{drift_idx}_{4*neighbor_distance + 1}x{4*neighbor_distance + 1}.npy"

            # Save numpy file.
            np.save(npy_filename, peak_window)

            # Dictionary with information about peaks.
            peak_data = {
                'label': label,
                'spectrum': spectrum.name,
                'ret_time': spectrum.ret_time[ret_idx],
                'drift_time': spectrum.drift_time[drift_idx],
                'intensity': spectrum.values[ret_idx, drift_idx],
                'npy_file': npy_filename,
            }
            spectrum_peak_data.append(peak_data)

        table_dir = output_dir / "table"
        if not table_dir.exists():
            table_dir.mkdir(parents=True, exist_ok=True)

        table_filename = table_dir / f"{spectrum.name}_peaks.xlsx"

        # Save the peak data as a .xlsx file
        peaks_dataframe = pd.DataFrame(spectrum_peak_data)
        peaks_dataframe.to_excel(table_filename, index=False)

        # Notify that the peak detection on this spectrum is finished and print the number of peaks.
        print(f"- {len(peak_coords)} Peaks saved - Spectrum {idx + 1} [{spectrum.name}]")

    else:
        # Notify that the peak detection on this spectrum is finished and print the number of peaks.
        print(f"- No Peaks saved - Spectrum {idx + 1} [{spectrum.name}]")


except Exception as err:
    print(f"Error - Spectrum {idx + 1} [{spectrum.name}] Skipped: {err=}, {type(err)=}")
    raise

- Peak Detection Started on: Spectrum 2
Found 17 peaks
 - Creating npy-files and heatmaps: Spectrum 2 [151207_144657]
- 17 Peaks saved - Spectrum 2 [151207_144657]


<Figure size 1200x800 with 0 Axes>

In [ ]:
import numpy as np
np.percentile(spectrum.values, [99.5])

# Inspect some tree parameters here ...
print(f"SPECTRUM INFOS")
print(f"Type: {type(spectrum.values)}")
print(f"Element Type: {type(spectrum.values[0, 0])}")
print("================================================")
print(f"Shape: {spectrum.values.shape}")
num_entries = spectrum.values.shape[0] * spectrum.values.shape[1]
num_non_0_entries = len(np.flatnonzero(spectrum.values))
print(f"Total Entries: {num_entries}")
print(f"Non-0 Enties: {num_non_0_entries}")
print(f"0 Enties: {num_entries - num_non_0_entries}")
print("================================================")
print(f"Max Value: {spectrum.values.max()}")
print(f"Min Value: {spectrum.values.min()}")
print("================================================")
over_50_values = len(np.where(non_normalized_spectrum.values >= 25)[0])
print(f"Total Values: {num_entries}")
print(f"Values > 50: {over_50_values}")
print(f"Values > 50 in % {(over_50_values/num_entries)*100}")
print(f"Percentile Value: {100 - (over_50_values/num_entries)*100}")



TypeError: 'findpeaks' object is not subscriptable